# SemMap · Text corpus → ConceptNet Local → Parquet

**Colab T4/L4 · режим NEW/RESUME · контрольные точки после каждого документа.**

Источник: [Txt.zip](https://drive.google.com/file/d/1I8dYdI4eLVmlhZjNewxl63EGgJKLmDqR/view). Результат: новая папка `text-corpus-<EXPERIMENT_LABEL>` внутри [целевой папки Colab](https://drive.google.com/drive/folders/1C7r8A0L0afrfq_8KTebOBM4BJqbZjXM4).

**Внимание:** ConceptNet Local поддерживает только английский; русский корпус будет отклонён. Требуется готовая локальная база SQLite ConceptNet Local — её файл Drive не указан. Текстовый этап рассчитан на CPU, наличие T4/L4 проверяется, GPU используется только следующими численными этапами semgraphex. На Drive хранятся результаты, расчёт выполняется на локальном SSD Colab.

Чтобы продолжить после прерывания, заново выполните **Run all** с тем же `EXPERIMENT_LABEL`, входным ZIP и базой SQLite. Ранее завершённые документы будут проверены и восстановлены. Документ, обработка которого оборвалась до контрольной точки, вычисляется снова.

In [ ]:
# Изменяемые параметры эксперимента
EXPERIMENT_LABEL = "semmap-text-01"  # Оставьте тем же для RESUME
SOURCE_ZIP_ID = "1I8dYdI4eLVmlhZjNewxl63EGgJKLmDqR"
RESULT_PARENT_ID = "1C7r8A0L0afrfq_8KTebOBM4BJqbZjXM4"
CONCEPTNET_DB_DRIVE_FILE_ID = ""  # УКАЖИТЕ ID готовой SQLite-базы ConceptNet Local
CONCEPTNET_DB_PATH = ""           # или путь до файла на смонтированном Drive
MAX_DOCS = 100                    # пилот: первые 100 .txt по имени; None = все
WORKERS = 0                       # 0 = авто по CPU и доступной RAM
THREADS_PER_WORKER = 1
MEMORY_LIMIT_PER_WORKER = "1GB"
MAX_EDGES_PER_DOC = 20_000_000
SPACY_MODEL = None                # при необходимости установите модель отдельно
REPO_BRANCH = "feature/text-corpus-colab-drive-resume"


In [ ]:
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from pathlib import Path
import os, shutil, subprocess, sys

credentials, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)
root_id = drive_api.files().get(fileId="root", fields="id").execute()["id"]

def mounted_drive_path(file_id):
    """Resolve actual MyDrive relative path by parent IDs; never guess file location."""
    parts = []
    current = file_id
    for _ in range(40):
        item = drive_api.files().get(
            fileId=current, fields="id,name,parents,mimeType,size"
        ).execute()
        parts.insert(0, item["name"])
        parents = item.get("parents", [])
        if not parents:
            if current != root_id:
                raise RuntimeError("Файл не расположен в My Drive; нужен доступный локальный путь")
            return Path("/content/drive/MyDrive").joinpath(*parts[1:])
        current = parents[0]
        if current == root_id:
            return Path("/content/drive/MyDrive").joinpath(*parts)
    raise RuntimeError("Слишком длинная цепочка родителей Google Drive")

archive_on_drive = mounted_drive_path(SOURCE_ZIP_ID)
output_parent = mounted_drive_path(RESULT_PARENT_ID)
if not archive_on_drive.is_file():
    raise FileNotFoundError(f"Нет доступа к ZIP в mount: {archive_on_drive}")
if not output_parent.is_dir():
    raise FileNotFoundError(f"Нет доступа к целевой папке: {output_parent}")
print("ZIP:", archive_on_drive, "size MB:", round(archive_on_drive.stat().st_size/1e6,1))
print("Output parent:", output_parent)


In [ ]:
import platform, psutil
try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU не назначен"
except ImportError:
    gpu_name = "PyTorch не установлен"
cpu = os.cpu_count() or 2
ram = psutil.virtual_memory().available
workers = (max(1, min(12, cpu - 1, int(ram // (2*1024**3)))))
if WORKERS:
    workers = WORKERS
print(f"Python {platform.python_version()} | CPU {cpu} | available RAM {ram/2**30:.1f} GiB | GPU {gpu_name}")
print("Workers:", workers, "| DuckDB threads/worker:", THREADS_PER_WORKER)
if sys.version_info < (3,12):
    raise RuntimeError("ConceptNet Local требует Python 3.12+; выберите подходящий Colab runtime")
if not (CONCEPTNET_DB_DRIVE_FILE_ID or CONCEPTNET_DB_PATH):
    raise RuntimeError(
        "Не найдена готовая база ConceptNet Local в доступном Drive. "
        "Укажите CONCEPTNET_DB_DRIVE_FILE_ID или CONCEPTNET_DB_PATH в первой ячейке.")
db_drive = (mounted_drive_path(CONCEPTNET_DB_DRIVE_FILE_ID)
            if CONCEPTNET_DB_DRIVE_FILE_ID else Path(CONCEPTNET_DB_PATH))
if not db_drive.is_file():
    raise FileNotFoundError(db_drive)
print("SQLite:", db_drive, round(db_drive.stat().st_size/1e9,2), "GB")


In [ ]:
import sqlite3
# Сначала проверяем, что это нужная база, не пытаясь модифицировать её.
with sqlite3.connect(db_drive.resolve().as_uri()+"?mode=ro", uri=True) as con:
    tables = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    assert {"concepts", "relations"}.issubset(tables), (
        f"Неверная база ConceptNet Local, таблицы: {tables}")
    indexes = con.execute("PRAGMA index_list('relations')").fetchall()
    print("Relations indexes:", [r[1] for r in indexes])
print("Кэш SQLite и ZIP будут размещены на локальном SSD, не на Drive.");


In [ ]:
from pathlib import Path
import shutil, subprocess
scratch_inputs = Path("/content/semmap-inputs")
scratch_inputs.mkdir(exist_ok=True)
db_local = scratch_inputs / "conceptnet.db"
if not db_local.exists() or db_local.stat().st_size != db_drive.stat().st_size:
    print("Staging indexed SQLite database to local SSD...")
    shutil.copyfile(db_drive, db_local)
else:
    print("Using existing local SQLite cache (full SHA verified by runner)")
print("Local DB:", db_local, round(db_local.stat().st_size/1e9,2),"GB")


In [ ]:
repo = Path("/content/semgraphex")
if not (repo / ".git").exists():
    subprocess.run(["git","clone","--single-branch","--branch",REPO_BRANCH,
                    "https://github.com/SemanticMap/semgraphex.git",str(repo)],check=True)
else:
    subprocess.run(["git","fetch","origin",REPO_BRANCH],cwd=repo,check=True)
    subprocess.run(["git","checkout",REPO_BRANCH],cwd=repo,check=True)
    subprocess.run(["git","reset","--hard","origin/"+REPO_BRANCH],cwd=repo,check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(repo)+"[text]"],check=True)
subprocess.run([sys.executable,"-m","nltk.downloader","stopwords"],check=True)
print("Code commit:",subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip())


## Запуск с автоматическим сохранением

Во время работы каждый завершённый документ записывается в собственную папку Drive, проверяется SHA-256, затем публикуется. Файл `progress.json` обновляется после каждой контрольной точки. Итоговый маркер `COMPLETED` появляется только после проверки **всех** результатов.

При повторном Run all **не меняйте** `EXPERIMENT_LABEL` и параметры данных. Смена числа процессов допустима. При ошибке программа не уничтожает сохранённые контрольные точки.

In [ ]:
cmd = [
    sys.executable, "-u", "-m", "semmap_haken.text_colab",
    "--archive", str(archive_on_drive),
    "--database", str(db_local),
    "--drive-parent", str(output_parent),
    "--label", EXPERIMENT_LABEL,
    "--workers", str(workers),
    "--threads-per-worker", str(THREADS_PER_WORKER),
    "--memory-limit", MEMORY_LIMIT_PER_WORKER,
    "--max-edges-per-document", str(MAX_EDGES_PER_DOC),
]
if MAX_DOCS is not None:
    cmd += ["--max-docs", str(MAX_DOCS)]
if SPACY_MODEL:
    cmd += ["--spacy-model", SPACY_MODEL]
print("Running:", " ".join(cmd), flush=True)
# Python's standard output streams live per-document checkpoints in Colab.
subprocess.run(cmd, cwd=repo, check=True)


In [ ]:
import json
exp_dir = output_parent / ("text-corpus-" + EXPERIMENT_LABEL)
print("Results:", exp_dir)
for name in ("progress.json", "manifest.json"):
    p = exp_dir / name
    if p.exists():
        print(name, json.dumps(json.loads(p.read_text()),ensure_ascii=False,indent=2)[:3500])
if (exp_dir / "COMPLETED").is_file():
    from semmap_haken.text_corpus import open_corpus
    db = open_corpus(exp_dir, verify=True)
    try:
        print("Documents:", db.execute("select count(*) from documents").fetchone()[0])
        print("Top relations:",db.execute(
            "select relation_uri, count(*) n from edges group by 1 order by n desc limit 10"
        ).fetchall())
    finally:
        db.close()
else:
    print("Корпус пока не завершён. Запустите Run all с той же меткой для RESUME.")
